# Step 4 — OpenRouter runner + smoke test

**What the runner guarantees**
- **Pinned endpoint**: each model is sent to exactly one OpenRouter endpoint (`provider.only`, `allow_fallbacks: false`), so quantization can't change mid-run.
- **Retries**: 429 / 5xx / timeouts / provider errors use exponential backoff. 401 / 402 / 403 stop the run at once (bad key, out of credits).
- **Cache**: `results/cache/<model>.jsonl`, keyed by (model, provider, exact messages, params). A rerun never pays twice.
- **Resume**: append-only `results/<dataset>/<model>__<method>.jsonl`. Already-done rows are skipped, and a half-written line from a crash is ignored.
- **Budget**: a hard USD cap on new spending per run.
- **Logged per row**: provider used, tokens in/out/reasoning, cost, latency, prompt version, params, raw output.

**Before the paid cell:** add `OPENROUTER_API_KEY` in Colab **Secrets** (🔑 icon in the left sidebar) and turn on notebook access.
Runtime: CPU.

In [1]:
# --- Setup: mount Drive, go to the project, install, import ---------------------------
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/LLM_as_Judge_Benchmark'
REPO_URL = 'https://github.com/moghbaie/LLM_as_Judge_Benchmark.git'

import os, sys, subprocess
if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
!git pull -q || true
!pip -q install -r requirements.txt
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))
import pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)

Mounted at /content/drive


## 1. Tests (offline: mocked HTTP and a fake judge, no key needed)

In [2]:
!python -m pytest -q tests/test_step4_runner.py

.......................                                                  [100%]
23 passed in 4.17s


## 2. Are the pinned endpoints live, and are prices unchanged? (public, no key)

In [3]:
from ljbench import models
from ljbench.runners.openrouter import check_pin, OpenRouterClient
pins = pd.DataFrame([check_pin(s) for s in models.enabled() if s.backend == 'openrouter'])
pins

,model,ok,problem,quantization,price_in,price_out
0,glm-5.3,True,None,fp8,1.40,4.40
1,deepseek-v4.1-flash,True,None,fp8,0.14,0.42
2,muse-spark-open,True,None,unknown,1.25,4.25
3,kimi-k3.1-pro,True,None,mxfp4,2.00,10.00
4,qwen3.8-max,True,None,unknown,2.00,6.00
5,claude-opus-5.5,True,None,unknown,4.00,20.00
6,claude-haiku-5.5,True,None,unknown,0.10,0.50
7,gpt-6-luna,True,None,unknown,0.10,0.50
8,gemini-3.8-flash,True,None,unknown,0.75,3.75


## 3. Key check (free: reads your key's limit and usage)

In [4]:
async with OpenRouterClient() as c:
    info = await c.key_info()
{k: info.get(k) for k in ('label', 'limit', 'limit_remaining', 'usage')}

{'label': 'sk-or-v1-862...cb9', 'limit': 50, 'limit_remaining': 50, 'usage': 0}

## 4. Smoke test: 20 held-out JudgeBench pairs × 2 orders × each model  (**paid, about $1 in total**)
Uses `judgebench_dev_20`, the prompt-debugging set, never the eval set. Set `RUN_PAID = True` to run it.

In [5]:
RUN_PAID = False
BUDGET_PER_MODEL = 0.50          # hard cap per model, USD
SMOKE_MODELS = ['deepseek-v4.1-flash', 'glm-5.3', 'muse-spark-open', 'qwen3.8-max', 'kimi-k3.1-pro']

from ljbench.run_api import run_one, score_dataset, RESULTS
from ljbench.runners.judge import RunConfig
summaries = []
if RUN_PAID:
    for key in SMOKE_MODELS:
        summaries.append(await run_one(key, 'judgebench_dev_20', 'm1', RunConfig(budget_usd=BUDGET_PER_MODEL)))
pd.DataFrame(summaries)

""


## 5. Look at real outputs

In [6]:
import json
from pathlib import Path
for f in sorted(Path(RESULTS, 'judgebench_dev_20').glob('*__m1.jsonl')):
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
    r = rows[0]
    print(f'\n#### {f.stem} | provider used: {r["provider_used"]} | gold {r["gold"]} -> label {r["label"]} '
          f'(conf {r["confidence"]}, parse {r["parse_mode"]}) | tokens out {r["tokens_out"]} '
          f'(reasoning {r["reasoning_tokens"]}) | ${r["cost_usd"]:.5f}')
    print('RAW:', r['raw_outputs'][-1][:600])

In [7]:
# Score (20 pairs is too few to rank models: this checks plumbing, parse rate and cost per call)
score_dataset('judgebench_dev_20').round(3)

""


## What to check before moving on (gate for Step 6)

| Check | Pass condition |
|---|---|
| Pins live | every row `ok=True` in cell 2 |
| Provider actually used | `provider_used` matches the pinned provider for every row |
| Parse failures | < 5% per model; otherwise look at the raw outputs and fix the prompt/parser **before** freezing |
| Output length | `tokens_out` / `reasoning_tokens` per call, which tells us whether reasoning mode must be capped (it drives cost) |
| Cost | multiply measured cost per call by 3,750 calls to get the real full-run budget |

Rerunning cell 4 costs **$0**: every response comes from the cache.